In [0]:
%run /Users/nethumgimsara605@gmail.com/ecommerce-lakehouse-databricks-repo/01-ingestion/setup-storage-connection

In [0]:
from pyspark.sql.functions import col, sum as spark_sum, count

In [0]:
silver_orders = spark.read.format("delta").load("abfss://silver@ecommercelakehouse01.dfs.core.windows.net/orders/")
silver_products = spark.read.format("delta").load("abfss://silver@ecommercelakehouse01.dfs.core.windows.net/products/")

print(f"Orders: {silver_orders.count()}, Products: {silver_products.count()}")

In [0]:
orders_with_products = silver_orders.join(
    silver_products.select("product_id", "product_name", "category"),
    on="product_id",
    how="inner"
)

print(f"Joined rows: {orders_with_products.count()}")

In [0]:
gold_top_products = orders_with_products.groupBy("product_id", "product_name", "category").agg(
    spark_sum("total_amount").alias("total_revenue"),
    spark_sum("quantity").alias("total_units_sold"),
    count("order_id").alias("total_orders")
)

gold_top_products.orderBy(col("total_revenue").desc()).show(10)

In [0]:
gold_top_products.write.format("delta").mode("overwrite").save("abfss://gold@ecommercelakehouse01.dfs.core.windows.net/top_products/")

In [0]:
gold_top_products = spark.read.format("delta").load("abfss://gold@ecommercelakehouse01.dfs.core.windows.net/top_products/")
print(f"Total rows: {gold_top_products.count()}")

gold_top_products.orderBy(col("total_revenue").desc()).show(10)
